# Entrainement des Modeles — Stray Dogs Tunisia
**ESPRIT — 3A IA | Ce notebook documente l'entraînement complet**

## Pourquoi des données synthétiques ?
Il n'existe pas de base de données officielle de comptage de chiens errants en Tunisie.
Cette approche Synthetic Data Generation est documentée en médecine vétérinaire (WHO, 2019).

**Hypothèses :**
- Ratio base : 1 chien / 12 habitants (OMS)
- Modulation par type de zone (commercial ×1.6, balnéaire ×1.4...)
- +2 chiens par POI alimentaire, +3 chiens par benne organique
- Bruit gaussien 15% pour la variabilité réelle

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent / 'src'))

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display
print('Imports OK')

Imports OK


## Partie 1 — Generation du dataset synthetique

In [2]:
from models.generate_synthetic_data import generate_full_dataset
df = generate_full_dataset()
print(f'Dataset : {len(df)} samples, {len(df.columns)} colonnes')
display(df[['gouvernorat','zone_type','population','nb_food_poi','nb_bennes_org','nb_chiens','risk_class']].head(10))

GÉNÉRATION DU DATASET SYNTHÉTIQUE
Couverture : 24 gouvernorats tunisiens (RGPH 2014)
Secteurs   : 24 × 50 = 1200 samples
  ✓ Tunis                : 50 secteurs générés (pop moy: 24,136)
  ✓ Ariana               : 50 secteurs générés (pop moy: 14,532)
  ✓ Ben Arous            : 50 secteurs générés (pop moy: 15,932)
  ✓ Manouba              : 50 secteurs générés (pop moy: 9,067)
  ✓ Nabeul               : 50 secteurs générés (pop moy: 17,261)
  ✓ Zaghouan             : 50 secteurs générés (pop moy: 3,912)
  ✓ Bizerte              : 50 secteurs générés (pop moy: 12,129)
  ✓ Beja                 : 50 secteurs générés (pop moy: 5,907)
  ✓ Jendouba             : 50 secteurs générés (pop moy: 10,460)
  ✓ Kef                  : 50 secteurs générés (pop moy: 6,146)
  ✓ Siliana              : 50 secteurs générés (pop moy: 5,026)
  ✓ Sousse               : 50 secteurs générés (pop moy: 15,173)
  ✓ Monastir             : 50 secteurs générés (pop moy: 11,865)
  ✓ Mahdia               : 50 secteurs 

,gouvernorat,zone_type,population,nb_food_poi,nb_bennes_org,nb_chiens,risk_class
0,Tunis,commercial,12115,19,20,1586,2
1,Tunis,residential,13415,5,18,965,1
2,Tunis,commercial,84741,32,124,14365,2
3,Tunis,park,15059,0,23,1349,1
4,Tunis,commercial,31712,32,31,5026,2
5,Tunis,commercial,33189,21,47,5790,2
6,Tunis,residential,22175,4,27,1349,1
7,Tunis,mixed,24028,21,30,2706,2
8,Tunis,industrial,31003,1,43,2632,1
9,Tunis,residential,9729,5,10,1140,2


In [3]:
# Analyse exploratoire
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor('#0d1117')
BG='#161b22'; TC='#c9d1d9'

def sax(ax, t):
    ax.set_facecolor(BG); ax.set_title(t, color='white', fontsize=10, fontweight='bold')
    ax.tick_params(colors=TC, labelsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
    ax.grid(True, color='#1a2040', alpha=0.3)

# Distribution nb_chiens
sax(axes[0,0], 'Distribution — Nb chiens')
axes[0,0].hist(df['nb_chiens'], bins=50, color='#58a6ff', alpha=0.8, edgecolor='#0d1117')
axes[0,0].axvline(df['nb_chiens'].median(), color='#f78166', lw=2, label=f'Mediane={df["nb_chiens"].median():.0f}')
axes[0,0].set_xlabel('Nb chiens', color=TC, fontsize=8)
axes[0,0].legend(facecolor=BG, labelcolor=TC, fontsize=8)

# Classes de risque
sax(axes[0,1], 'Classes de risque')
counts = df['risk_class'].value_counts().sort_index()
axes[0,1].bar(['Faible','Moyen','Eleve'], counts.values, color=['#2ecc71','#f39c12','#e74c3c'], alpha=0.85)
for i,v in enumerate(counts.values):
    axes[0,1].text(i, v+5, f'{v} ({v/len(df)*100:.0f}%)', ha='center', color='white', fontsize=8)

# Population vs Chiens
sax(axes[0,2], 'Population vs Chiens')
for cls, col, lbl in [(0,'#2ecc71','Faible'),(1,'#f39c12','Moyen'),(2,'#e74c3c','Eleve')]:
    sub = df[df['risk_class']==cls]
    axes[0,2].scatter(sub['population'], sub['nb_chiens'], c=col, s=8, alpha=0.4, label=lbl)
axes[0,2].set_xlabel('Population', color=TC, fontsize=8)
axes[0,2].set_ylabel('Nb chiens', color=TC, fontsize=8)
axes[0,2].legend(facecolor=BG, labelcolor=TC, fontsize=7)

# Zone type vs chiens
sax(axes[1,0], 'Type de zone vs Chiens')
zone_stats = df.groupby('zone_type')['nb_chiens'].median().sort_values(ascending=False)
axes[1,0].barh(zone_stats.index, zone_stats.values, color='#d2a8ff', alpha=0.85)
axes[1,0].set_xlabel('Mediane chiens', color=TC, fontsize=8)

# Top gouvernorats
sax(axes[1,1], 'Chiens medianes par gouvernorat (top 8)')
gov_med = df.groupby('gouvernorat')['nb_chiens'].median().nlargest(8)
axes[1,1].barh(gov_med.index, gov_med.values, color='#58a6ff', alpha=0.85)
axes[1,1].set_xlabel('Mediane chiens', color=TC, fontsize=8)

# Correlations
sax(axes[1,2], 'Correlations avec nb_chiens')
num_cols = ['population','nb_menages','nb_food_poi','nb_bennes_org','zone_coeff','densite_pop']
corrs = df[num_cols].corrwith(df['nb_chiens']).sort_values()
axes[1,2].barh(corrs.index, corrs.values, color=['#e74c3c' if v>0 else '#3498db' for v in corrs], alpha=0.85)
axes[1,2].axvline(0, color='white', lw=0.8)
axes[1,2].set_xlabel('Correlation Pearson', color=TC, fontsize=8)

plt.suptitle('Dataset Synthetique — 1200 secteurs | 24 gouvernorats tunisiens',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/dataset_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Analyse sauvegardee dans visualizations/dataset_analysis.png')

Analyse sauvegardee dans visualizations/dataset_analysis.png


## Partie 2 — Entrainement complet avec metriques

In [4]:
from models.train_models import (
    load_data, prepare_data,
    train_rf_regressor, train_rf_classifier, train_gb_regressor,
    plot_training_results, save_metadata, FEATURE_COLS
)

df_t = load_data()
(X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test, scaler) = prepare_data(df_t)
print(f'Train: {len(X_train)} | Test: {len(X_test)} | Features: {len(FEATURE_COLS)}')

[TRAIN] Dataset chargé : 1,200 samples, 9 features
[TRAIN] Train : 960 | Test : 240
Train: 960 | Test: 240 | Features: 9


In [5]:
print('=== 1/3 — Random Forest Regressor ===')
rf_reg, importances = train_rf_regressor(X_train, X_test, y_reg_train, y_reg_test, evaluate=True)

=== 1/3 — Random Forest Regressor ===

[TRAIN] ── Random Forest Regressor ──
  MAE  : 175.0 chiens
  RMSE : 280.7 chiens
  R²   : 0.9319
  CV R²: 0.9005 ± 0.0321

  Feature importance (top 5) :
    population                : 0.8126
    nb_bennes_org             : 0.0506
    nb_food_poi               : 0.0487
    zone_coeff                : 0.0272
    nb_menages                : 0.0262

  ✅ Sauvegardé : models/trained/rf_regressor.joblib


In [6]:
print('=== 2/3 — Random Forest Classifier ===')
rf_clf = train_rf_classifier(X_train, X_test, y_clf_train, y_clf_test, evaluate=True)

=== 2/3 — Random Forest Classifier ===

[TRAIN] ── Random Forest Classifier (risque) ──
  Accuracy : 0.8583

  Rapport de classification :
              precision    recall  f1-score   support

      Faible      0.000     0.000     0.000         2
       Moyen      0.873     0.886     0.879       140
       Élevé      0.837     0.837     0.837        98

    accuracy                          0.858       240
   macro avg      0.570     0.574     0.572       240
weighted avg      0.851     0.858     0.855       240

  CV Accuracy: 0.8208 ± 0.0327

  ✅ Sauvegardé : models/trained/rf_classifier.joblib


In [7]:
print('=== 3/3 — Gradient Boosting Regressor ===')
gb_reg = train_gb_regressor(X_train, X_test, y_reg_train, y_reg_test)

=== 3/3 — Gradient Boosting Regressor ===

[TRAIN] ── Gradient Boosting Regressor ──
  MAE  : 163.8 chiens
  RMSE : 266.6 chiens
  R²   : 0.9385

  ✅ Sauvegardé : models/trained/gb_regressor.joblib


In [8]:
# Graphiques d'entrainement
plot_training_results(rf_reg, gb_reg, rf_clf, X_train, X_test,
                      y_reg_train, y_reg_test, y_clf_test, importances)

fig, ax = plt.subplots(figsize=(20,12), facecolor='#0d1117')
ax.imshow(mpimg.imread('../visualizations/training_results.png'))
ax.axis('off'); plt.tight_layout(); plt.show()


[TRAIN] Visualisation sauvegardée : C:\Users\azizb\Downloads\stray_dogs_v2_complet\stray_dogs_v2\visualizations\training_results.png


In [9]:
# Comparaison RF vs GB vs Ensemble
from sklearn.metrics import mean_absolute_error, r2_score

y_rf = rf_reg.predict(X_test)
y_gb = gb_reg.predict(X_test)
y_en = y_rf * 0.6 + y_gb * 0.4

comp = pd.DataFrame({
    'Modele': ['Random Forest', 'Gradient Boosting', 'Ensemble (RF×0.6 + GB×0.4)'],
    'MAE':    [mean_absolute_error(y_reg_test, y_rf),
               mean_absolute_error(y_reg_test, y_gb),
               mean_absolute_error(y_reg_test, y_en)],
    'RMSE':   [(((y_reg_test-y_rf)**2).mean())**0.5,
               (((y_reg_test-y_gb)**2).mean())**0.5,
               (((y_reg_test-y_en)**2).mean())**0.5],
    'R2':     [r2_score(y_reg_test, y_rf),
               r2_score(y_reg_test, y_gb),
               r2_score(y_reg_test, y_en)],
}).set_index('Modele').round(4)

print('Comparaison des modeles :')
display(comp)
print('\nLe pipeline final utilise l\'Ensemble pour la meilleure precision.')

Comparaison des modeles :


,MAE,RMSE,R2
Modele,,,
Random Forest,175.0304,280.6526,0.9319
Gradient Boosting,163.8456,266.6222,0.9385
Ensemble (RF×0.6 + GB×0.4),168.0236,270.3687,0.9368



Le pipeline final utilise l'Ensemble pour la meilleure precision.


In [ ]:
# Resume final 
import json
meta = save_metadata(rf_reg, gb_reg, rf_clf, X_test, y_reg_test, y_clf_test, scaler)

print('='*55)
print('  RESUME — SOUTENANCE')
print('='*55)
print(f"""
  Dataset synthetique   : 1200 samples / 24 gouvernorats
  Split train/test      : 80% / 20%
  Features              : {len(FEATURE_COLS)}

  RF Regressor R2       : {meta['rf_regressor']['r2']}
  RF Regressor MAE      : {meta['rf_regressor']['mae']} chiens
  GB Regressor R2       : {meta['gb_regressor']['r2']}
  RF Classifier Acc     : {meta['rf_classifier']['accuracy']}

  Modeles sauvegardes   : models/trained/
  rf_regressor.joblib, gb_regressor.joblib,
  rf_classifier.joblib, scaler.joblib
""")

[TRAIN] Métadonnées sauvegardées : C:\Users\azizb\Downloads\stray_dogs_v2_complet\stray_dogs_v2\models\trained\training_metadata.json
  RESUME — SOUTENANCE

  Dataset synthetique   : 1200 samples / 24 gouvernorats
  Split train/test      : 80% / 20%
  Features              : 9

  RF Regressor R2       : 0.9319
  RF Regressor MAE      : 175.03 chiens
  GB Regressor R2       : 0.9385
  RF Classifier Acc     : 0.8583

  Modeles sauvegardes   : models/trained/
  rf_regressor.joblib, gb_regressor.joblib,
  rf_classifier.joblib, scaler.joblib

